In [ ]:
import torch
#!pip install deep_translator
#!pip install emoji
from google.colab import files
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.optim import AdamW
from transformers import get_scheduler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import random
import re
import emoji
import nltk
from nltk.corpus import wordnet
from deep_translator import GoogleTranslator
import time
nltk.download('wordnet')


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 15
BATCH_SIZE = 16
LR = 3e-5


# Early stopping utility
class EarlyStopping:
    def __init__(self, patience=3, delta=0):
        self.patience = patience
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.delta = delta
    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0


# Preprocessing function with emoji demojization
def preprocess(text):
    text = emoji.demojize(text)
    text = re.sub(r':', ' ', text)
    text = re.sub(r'_', ' ', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = text.lower().strip()
    return text


# Synonym replacement augmentation
def synonym_replacement(sentence, n=2):
    words = sentence.split()
    new_words = words.copy()
    candidates = list(set([w for w in words if wordnet.synsets(w)]))
    random.shuffle(candidates)
    num_replaced = 0
    for word in candidates:
        synsets = wordnet.synsets(word)
        if not synsets:
            continue
        lemmas = synsets[0].lemmas()
        if not lemmas:
            continue
        synonym = lemmas[0].name()
        if synonym != word:
            new_words = [synonym if w == word else w for w in new_words]
            num_replaced += 1
        if num_replaced >= n:
            break
    return ' '.join(new_words)


# Back-translation augmentation
def back_translate(text, src='en', mid='fr', max_retries=3, delay=2):
    for attempt in range(max_retries):
        try:
            trans = GoogleTranslator(source=src, target=mid).translate(text)
            back = GoogleTranslator(source=mid, target=src).translate(trans)
            return back
        except Exception:
            time.sleep(delay)
    return text


# Dataset Class
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(text, padding='max_length', truncation=True, max_length=self.max_len, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }


# Focal loss for imbalance
class FocalLoss(torch.nn.Module):
    def __init__(self, gamma=2, weight=None):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.weight = weight
        self.ce = torch.nn.CrossEntropyLoss(weight=weight)
    def forward(self, inputs, targets):
        logpt = -self.ce(inputs, targets)
        pt = torch.exp(logpt)
        loss = ((1 - pt)**self.gamma) * (-logpt)
        return loss.mean()


# Load and preprocess data
#uploaded = files.upload() # for google colab
df = pd.read_csv('sample_for_labeling.csv')
df['text'] = df['text'].apply(preprocess)


from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['label'])
NUM_LABELS = len(le.classes_)


# Define minority and hard classes
minority_classes = ['anger', 'relief'] # minority classes
hard_classes = ['confusion', 'longing_regret', 'sadness'] # classes with low performance


# Augmentation function for a sentence multiple times
def augment_text(text, n_rep=2, n_bt=1):
    augmented = []
    for _ in range(n_rep):
        augmented.append(synonym_replacement(text))
    for _ in range(n_bt):
        bt_text = back_translate(text)
        if bt_text != text:
            augmented.append(bt_text)
    return augmented


# Augment data
aug_texts = []
aug_labels = []

#Data Augmentation Loop
for idx, row in df.iterrows():
    aug_texts.append(row['text'])
    aug_labels.append(row['label_enc'])
    label_name = row['label']
    if label_name in minority_classes:
       augmented_texts = augment_text(row['text'], n_rep=3, n_bt=2)
       aug_texts.extend(augmented_texts)
       aug_labels.extend([row['label_enc']] * len(augmented_texts))
    elif label_name in hard_classes:
         augmented_texts = augment_text(row['text'], n_rep=2, n_bt=1)
         aug_texts.extend(augmented_texts)
         aug_labels.extend([row['label_enc']] * len(augmented_texts))



df_aug = pd.DataFrame({'text': aug_texts, 'label_enc': aug_labels})


# Stratified K-Fold Cross Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def train_model(model_name, train_texts, train_labels, val_texts, val_labels):
    print(f"Training {model_name}...")


    if model_name == 'bert':
        tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=NUM_LABELS)
    else:
        tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
        model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=NUM_LABELS)


    model.to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LR)
    train_dataset = SentimentDataset(train_texts, train_labels, tokenizer)
    val_dataset = SentimentDataset(val_texts, val_labels, tokenizer)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)


    num_training_steps = EPOCHS * len(train_loader)
    lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)


    # Class weights based on training labels for focal loss
    class_counts = np.bincount(train_labels)
    class_weights = 1. / class_counts
    weights = torch.FloatTensor(class_weights).to(DEVICE)
    loss_fn = FocalLoss(gamma=2, weight=weights)


    early_stopper = EarlyStopping(patience=3)


    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        loop = tqdm(train_loader, desc=f'Epoch {epoch+1}')
        for batch in loop:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)


            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits


            loss = loss_fn(logits, labels)
            loss.backward()
            optimizer.step()
            lr_scheduler.step()


            total_loss += loss.item()
            loop.set_postfix(loss=loss.item())


        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} average loss: {avg_loss:.4f}")


        model.eval()
        val_preds = []
        val_true = []
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)


                outputs = model(input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                val_loss += loss_fn(logits, labels).item()
                val_preds.extend(torch.argmax(logits, axis=1).cpu().numpy())
                val_true.extend(labels.cpu().numpy())


        avg_val_loss = val_loss / len(val_loader)
        print(f"Validation loss: {avg_val_loss:.4f}")
        print(classification_report(val_true, val_preds, target_names=le.classes_))


        if early_stopper(avg_val_loss):
            print("Early stopping triggered.")
            break


# Run cross-validation on both models
for model_key in ['distilbert', 'bert']:
    print(f"\n===== Starting cross-validation for {model_key.upper()} =====")
    for fold, (train_idx, val_idx) in enumerate(skf.split(df_aug['text'], df_aug['label_enc'])):
        print(f"\n--- Fold {fold + 1} ---")
        train_texts = df_aug.loc[train_idx, 'text'].tolist()
        train_labels = df_aug.loc[train_idx, 'label_enc'].tolist()
        val_texts = df_aug.loc[val_idx, 'text'].tolist()
        val_labels = df_aug.loc[val_idx, 'label_enc'].tolist()
        train_model(model_key, train_texts, train_labels, val_texts, val_labels)

# ======= Retrain final DistilBERT model on full dataset for deployment ======== #

final_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
final_model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=NUM_LABELS)
final_model.to(DEVICE)

full_train_dataset = SentimentDataset(df_aug['text'].tolist(), df_aug['label_enc'].tolist(), final_tokenizer)
full_train_loader = DataLoader(full_train_dataset, batch_size=BATCH_SIZE, shuffle=True)

final_optimizer = AdamW(final_model.parameters(), lr=LR)
final_num_steps = EPOCHS * len(full_train_loader)
final_scheduler = get_scheduler("linear", optimizer=final_optimizer, num_warmup_steps=0, num_training_steps=final_num_steps)

# Compute class weights on full dataset labels
class_counts = np.bincount(df_aug['label_enc'])
class_weights = 1. / class_counts
weights = torch.FloatTensor(class_weights).to(DEVICE)

final_loss_fn = FocalLoss(gamma=2, weight=weights)
final_model.train()

for epoch in range(EPOCHS):
    total_loss = 0
    loop = tqdm(full_train_loader, desc=f'FINAL MODEL Epoch {epoch+1}')
    for batch in loop:
        final_optimizer.zero_grad()
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        outputs = final_model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        loss = final_loss_fn(logits, labels)
        loss.backward()
        final_optimizer.step()
        final_scheduler.step()
        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    avg_loss = total_loss / len(full_train_loader)
    print(f"Final DistilBERT epoch {epoch+1} avg loss: {avg_loss:.4f}")

# Save final model and tokenizer for inference deployment
save_directory = 'final_distilbert_sentiment_model'
final_model.save_pretrained(save_directory)
final_tokenizer.save_pretrained(save_directory)
print(f"Final model and tokenizer saved to: {save_directory}")
